# Library & Archival Extraction Pipeline

End-to-end extraction pipeline for digitized library materials — books,
manuscripts, sheet music, newspapers, maps, correspondence, ephemera,
multilingual archival content.

This notebook is a library-focused variant of `test_extraction_pipeline.ipynb`
(which is tuned for grant administration documents).

**Pipeline:**
1. Connect to shared vLLM endpoint (remote mode) or load model locally
2. Upload sample library scans to `/ocr/`
3. Per-page extraction via 3-page sliding window — VLM produces library
   schema JSON with bibliographic metadata, full verbatim transcription,
   marginalia, stamps/bookplates, musical notation, physical condition
4. Document-level synthesis (pass 2) produces catalog-style metadata
5. Outputs: `*_extracted.jsonl` (audit) + `*_final.json` (canonical)

**Schema differences vs grant admin pipeline:**
- bibliographic block (title, creator, date, language, publisher)
- body_text (full verbatim page transcription)
- structure (headings, footnotes, lists, poetry_verse)
- visual_elements (illustrations, musical_notation, maps)
- marginalia (handwritten annotations with location + writing instrument)
- stamps_marks (library stamps, bookplates, accession numbers, barcodes)
- physical_condition (damage, legibility, scan issues)
- page_type (book_page, title_page, manuscript, sheet_music, newspaper, etc.)


In [ ]:
from pathlib import Path
import os

# ── VLM mode: "remote" (shared vLLM endpoint) or "local" (load on this GPU) ──
VLM_MODE = "remote"

# Remote mode settings (vLLM endpoint)
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://qwen3--vl--32b--instruct-awq-01.runai-shared-models.svc.cluster.local/v1")
VLLM_MAX_TOKENS = 4096

# Model name — used for output metadata. In remote mode, auto-detected from endpoint.
MODEL_NAME = os.environ.get("VLM_MODEL", "QuantTrio/Qwen3-VL-32B-Instruct-AWQ")

if VLM_MODE == "remote":
    import httpx
    # Auto-detect model name from vLLM endpoint
    try:
        resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=5)
        resp.raise_for_status()
        models = resp.json().get("data", [])
        if models:
            MODEL_NAME = models[0]["id"]
        print(f"Remote VLM endpoint: {VLLM_BASE_URL}")
        print(f"Model: {MODEL_NAME}")
        print("Status: CONNECTED")
    except Exception as e:
        print(f"Remote VLM endpoint: {VLLM_BASE_URL}")
        print(f"Status: UNREACHABLE ({e})")
        print("\nFix: update VLLM_BASE_URL above, or switch to VLM_MODE = \"local\"")
else:
    # Local mode: set HF cache paths for model loading
    os.environ["HF_HOME"] = "/models/.cache/huggingface"
    os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
    os.environ["HF_HUB_OFFLINE"] = "1"

    models_dir = Path("/models/.cache/huggingface")
    if models_dir.exists():
        model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
        print(f"Local mode. Shared models PVC: {len(model_dirs)} model(s)")
        has_vlm = any("Qwen3-VL" in d or "Qwen2.5-VL" in d for d in model_dirs)
        if not has_vlm:
            print("WARNING: No Qwen VL model found!")
    else:
        print("WARNING: /models/.cache/huggingface not found.")


### GPU check (local mode only)

Runs `nvidia-smi` when `VLM_MODE = "local"` so you can confirm the workspace has a GPU and see its memory. Silently skipped in remote mode since the workspace is CPU-only.


In [ ]:
if VLM_MODE == "local":
    import subprocess
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
             "--format=csv,noheader"],
            text=True, stderr=subprocess.STDOUT)
        print(out)
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f"nvidia-smi not available: {e}")
        print("Local mode requires GPU access — check workspace compute settings.")
else:
    print("Remote mode — GPU check skipped (vLLM server has the GPU, workspace is CPU-only)")

## 2. Load the model (local mode only)

**Skip this cell if using `VLM_MODE = "remote"`** — the model is already loaded on the vLLM server.

### Requirements for local mode

If you want to run the model directly in the notebook, the workspace needs:

1. **GPU fraction on the workspace itself** — edit the workspace in RunAI UI:
   - `25%` of a 96 GB GPU for the AWQ 4-bit model (~20 GB VRAM)
   - `75-85%` for the full bf16 model (~64 GB VRAM)

2. **Shared models PVC mounted** at `/models` — add `shared-models` data volume in the workspace config

3. **HF env vars** set on the workspace:
   - `HF_HOME=/models/.cache/huggingface`
   - `HF_HUB_CACHE=/models/.cache/huggingface`
   - `HF_HUB_OFFLINE=1` (prevents accidental downloads)

4. **Model weights already on the PVC.** Before running the load cell, verify:

   ```bash
   ls /models/.cache/huggingface/ | grep Qwen
   ```

   You should see a directory matching your chosen `MODEL_NAME`. If not, the
   model needs to be downloaded first — see [Setup Data Volumes](../docs/setup-data-volumes.md)
   for download instructions (run from the `shared-models` project).

5. **`transformers` + `qwen-vl-utils` installed** — the install cell below handles this
   automatically when `VLM_MODEL = "local"`.

Change `MODEL_NAME` below to switch models. The `Auto` class handles both Qwen2.5-VL and Qwen3-VL architectures automatically.


In [ ]:
# Skip this cell in remote mode — model is on the vLLM server
if VLM_MODE == "local":
    import time
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

    # ── Choose a model (uncomment one) ─────────────────────────────────
    # MODEL_NAME = "Qwen/Qwen3-VL-32B-Instruct"          # ~64 GB bf16
    MODEL_NAME = "QuantTrio/Qwen3-VL-32B-Instruct-AWQ"  # ~20 GB INT4 — default
    # MODEL_NAME = "Qwen/Qwen2.5-VL-72B-Instruct-AWQ"  # ~40 GB INT4
    # MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"          # ~16 GB bf16

    USE_BNB_4BIT = False
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ) if USE_BNB_4BIT else None

    print(f"Loading {MODEL_NAME}...")
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",
        quantization_config=bnb_config,
    )
    elapsed = time.time() - t0
    print(f"Model loaded in {elapsed:.1f}s — Device: {model.device}")

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            used = torch.cuda.memory_allocated(i) / 1024**3
            print(f"GPU {i}: {used:.1f} / {total:.1f} GB")
else:
    print("Remote mode — skipping local model load.")
    print(f"Using vLLM endpoint: {VLLM_BASE_URL}")

    # Smoke test: send a trivial request to confirm the endpoint responds
    import time, httpx
    print("\nRunning smoke test...")
    t0 = time.time()
    try:
        resp = httpx.post(
            f"{VLLM_BASE_URL}/chat/completions",
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": "Say OK."}],
                "max_tokens": 10,
                "temperature": 0.0,
            },
            timeout=60,
        )
        resp.raise_for_status()
        reply = resp.json()["choices"][0]["message"]["content"].strip()
        elapsed = time.time() - t0
        print(f"  Reply: {reply!r}")
        print(f"  Elapsed: {elapsed:.1f}s")
        print(f"  Smoke test PASSED — endpoint is live and generating.")
    except Exception as e:
        print(f"  Smoke test FAILED after {time.time()-t0:.1f}s: {e}")
        print(f"  Check that the vLLM job is Running at {VLLM_BASE_URL}")


### Install local-mode dependencies (local mode only)

Skip this cell in remote mode — all base packages are already installed by the workspace startup.

For local mode, this installs `transformers`, `qwen-vl-utils`, and `autoawq` if not already present.


In [ ]:
# Local mode needs extra packages (transformers, qwen-vl-utils, autoawq)
# beyond what the workspace startup installs. Skip this cell in remote mode.

if VLM_MODE == "local":
    import importlib.util
    local_deps = ["transformers", "qwen_vl_utils", "autoawq"]
    missing = [p for p in local_deps if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing local mode deps: {missing}")
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--break-system-packages", "-q"] + missing)
        print("Done.")
    else:
        print("Local mode packages OK: transformers, qwen-vl-utils, autoawq")
else:
    print("Remote mode — nothing to install here.")


### Define helper functions

- `run_vlm(messages)` — core inference: dispatches to remote vLLM endpoint or local transformers model based on `VLM_MODE`
- `extract_page(image, prompt)` — encodes a page image as base64 PNG and sends it with the extraction prompt to the VLM. Supports sliding window context (prev/next page images).
- `extract_links(page)` — extracts hyperlinks from a PDF page's metadata layer

These functions work identically in both remote and local mode — the only difference is where the model runs.


In [ ]:
import base64, io, time

def _encode_image_b64(image):
    """Encode a PIL Image as a base64 PNG data URI."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

def _run_vlm_remote(messages, max_tokens=4096):
    """Send messages to the vLLM OpenAI-compatible API."""
    import httpx

    # Convert messages to OpenAI format (base64 images stay as-is)
    oai_messages = []
    for msg in messages:
        oai_content = []
        for part in msg["content"]:
            if part["type"] == "text":
                oai_content.append({"type": "text", "text": part["text"]})
            elif part["type"] == "image":
                image_url = part["image"]
                oai_content.append({"type": "image_url", "image_url": {"url": image_url}})
        oai_messages.append({"role": msg["role"], "content": oai_content})

    payload = {
        "model": MODEL_NAME,
        "messages": oai_messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        # Anti-repetition — VLMs sometimes get stuck in loops when they
        # can't transcribe decorative/stylized elements. These push it
        # to diversify tokens and break out of repetition.
        "frequency_penalty": 0.3,
        "repetition_penalty": 1.05,
        # Force valid JSON output at the token level. vLLM enforces this
        # during generation so we do not get mixed quote styles, JS-style
        # comments, or trailing garbage.
        "response_format": {"type": "json_object"},
    }
    resp = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=300,  # 5 min — large pages can be slow
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

def _run_vlm_local(messages, max_tokens=4096):
    """Run inference on the locally loaded model."""
    from qwen_vl_utils import process_vision_info

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def run_vlm(messages, max_tokens=4096):
    """Run VLM inference — dispatches to remote (vLLM) or local (transformers)."""
    if VLM_MODE == "remote":
        return _run_vlm_remote(messages, max_tokens)
    else:
        return _run_vlm_local(messages, max_tokens)

def extract_links(page):
    """Extract hyperlinks from a PDF page. Returns list of {text, url}."""
    links = []
    for link in page.get_links():
        uri = link.get("uri", "")
        if not uri:
            continue
        rect = link.get("from", None)
        text = page.get_text("text", clip=rect).strip() if rect else ""
        links.append({"text": text, "url": uri})
    return links

def extract_page(image, prompt, max_tokens=4096, filename=None, links=None,
                 prev_image=None, next_image=None):
    """Send a rendered page image to the VLM for structured extraction.

    Sliding window: if prev_image / next_image are provided, they are
    included as visual context so the VLM can detect cross-page content.
    The VLM extracts data from the CENTER page only.
    """
    context_parts = []
    if filename:
        context_parts.append(f"Source filename: {filename}")
    if links:
        link_lines = "\n".join(f"  {l['text']} -> {l['url']}" for l in links)
        context_parts.append(f"Hyperlinks found in this page:\n{link_lines}")
    if prev_image or next_image:
        context_parts.append(
            "CONTEXT: Surrounding pages are shown for context only. "
            "Extract data from the CENTER PAGE only. Use the surrounding "
            "pages to determine if content continues across page boundaries "
            "(set continuation flags accordingly)."
        )
    prefix = "\n\n".join(context_parts)
    text = f"{prefix}\n\n{prompt}" if prefix else prompt

    content = []
    if prev_image:
        content.append({"type": "text", "text": "[PREVIOUS PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(prev_image)})
    content.append({"type": "text", "text": "[CENTER PAGE — extract this page]"})
    content.append({"type": "image", "image": _encode_image_b64(image)})
    if next_image:
        content.append({"type": "text", "text": "[NEXT PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(next_image)})
    content.append({"type": "text", "text": text})

    messages = [{"role": "user", "content": content}]
    return run_vlm(messages, max_tokens)


# Character-level JSON cleanup for VLM responses
_CURLY_QUOTES = {
    "“": "\"", "”": "\"",   # left/right double curly quotes
    "‘": "'",  "’": "'",     # left/right single curly quotes
    "„": "\"", "‟": "\"",   # double low-9 / double high-reversed-9 quotes
    "‹": "'",  "›": "'",     # single angle quotes
    "«": "\"", "»": "\"",   # guillemets (rare in VLM output)
}


def _normalize_vlm_json(text: str) -> str:
    """Normalize common quote/formatting quirks before JSON parsing."""
    # Replace curly/smart quotes with straight quotes
    for bad, good in _CURLY_QUOTES.items():
        text = text.replace(bad, good)
    # Strip JS-style // line comments (VLMs sometimes add these)
    import re
    text = re.sub(r"(?m)\s*//[^\n]*$", "", text)
    # Fix escaped quotes at positions where unescaped are expected.
    # This is a common VLM slip: backslash-escaping every quote even outside
    # strings. Replace \" → " when it appears right after a comma/brace/
    # newline (positions where JSON expects a key/value start).
    text = re.sub(r'([,\{\[\n\r\s])\\"', r'\1"', text)
    return text


def parse_vlm_json(raw: str):
    """Robustly parse a VLM response as JSON.

    Handles common issues:
    - Markdown fences (```json ... ``` or ``` ... ```)
    - Leading/trailing prose
    - Truncated output (fallback to extract {...} substring)
    - Runaway repetition (VLM stuck in a loop) — truncate at repetition,
      rebuild JSON from last balanced brace

    Returns (parsed_dict, error_msg_or_None).
    """
    if not raw or not raw.strip():
        return None, "empty response"

    cleaned = raw.strip()

    # Strip markdown fences
    if cleaned.startswith("```"):
        first_newline = cleaned.find("\n")
        if first_newline != -1:
            cleaned = cleaned[first_newline + 1:]
        cleaned = cleaned.rsplit("```", 1)[0].strip()

    # Normalize common VLM quirks (curly quotes, JS comments, escaped quotes)
    cleaned = _normalize_vlm_json(cleaned)

    # Try direct parse
    first_err_msg = None
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        first_err_msg = str(e)

    # Fallback 1a: blanket-unescape any remaining \" -> ".
    # If the VLM started escaping quotes mid-response (e.g. opens with
    # "key": "value", then shifts to \"key\": \"value\"), this recovers.
    # Only attempt if the string CONTAINS escaped quotes at all.
    if '\\"' in cleaned:
        unescaped = cleaned.replace('\\"', '"')
        try:
            return json.loads(unescaped), None
        except json.JSONDecodeError:
            # Also try on the first-brace-to-last-brace slice
            fb = unescaped.find("{")
            lb = unescaped.rfind("}")
            if fb != -1 and lb > fb:
                try:
                    return json.loads(unescaped[fb:lb + 1]), None
                except json.JSONDecodeError:
                    pass

    # Fallback 1: first { ... last }
    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")
    if first_brace != -1 and last_brace > first_brace:
        try:
            return json.loads(cleaned[first_brace:last_brace + 1]), None
        except json.JSONDecodeError:
            pass

    # Fallback 1b: per-field recovery.
    # If a single field in the middle is malformed but other fields are valid,
    # parse field-by-field and keep only the good ones. Drops just the bad
    # field instead of everything after it.
    def _recover_by_field(text):
        for candidate_text in (text, text.replace('\\"', '"')):
            fb = candidate_text.find("{")
            lb = candidate_text.rfind("}")
            if fb == -1 or lb <= fb:
                continue
            body = candidate_text[fb + 1:lb]

            # Split body into field segments at depth-0 commas (relative to body)
            segments = []
            depth = 0
            in_string = False
            escape = False
            last_start = 0
            for idx, ch in enumerate(body):
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"':
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if ch in "{[":
                    depth += 1
                elif ch in "}]":
                    depth -= 1
                elif ch == "," and depth == 0:
                    segments.append(body[last_start:idx])
                    last_start = idx + 1
            segments.append(body[last_start:])

            # Test each segment; keep parseable ones
            valid_segments = []
            for seg in segments:
                seg = seg.strip()
                if not seg:
                    continue
                try:
                    json.loads("{" + seg + "}")
                    valid_segments.append(seg)
                except json.JSONDecodeError:
                    continue

            if not valid_segments:
                continue
            reconstructed = "{" + ",".join(valid_segments) + "}"
            try:
                return json.loads(reconstructed)
            except json.JSONDecodeError:
                continue
        return None

    field_recovered = _recover_by_field(cleaned)
    if field_recovered is not None:
        return field_recovered, None

    # Fallback 1c: truncate busted tail.
    # VLMs often produce valid JSON for all the content fields, then garble
    # the last 1-2 fields (continuation, other_metadata). Walk back from
    # the end looking for commas where we can close the JSON with "}" and
    # get a valid parse. Drops the garbled tail fields — they are usually
    # non-critical (continuation is backed up by programmatic detection).
    def _try_truncate_tail(text):
        # Also try on the unescaped version
        for candidate_text in (text, text.replace('\\"', '"')):
            fb = candidate_text.find("{")
            if fb == -1:
                continue
            body = candidate_text[fb:]
            # Find all comma positions at depth 1 (outermost object)
            depth = 0
            in_string = False
            escape = False
            comma_positions = []
            for idx, ch in enumerate(body):
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"' and not escape:
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if ch == "{" or ch == "[":
                    depth += 1
                elif ch == "}" or ch == "]":
                    depth -= 1
                elif ch == "," and depth == 1:
                    comma_positions.append(idx)
            # Try truncating at each comma position from latest to earliest
            for pos in reversed(comma_positions):
                candidate = body[:pos] + "}"
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    continue
        return None

    truncated = _try_truncate_tail(cleaned)
    if truncated is not None:
        return truncated, None

    # Fallback 2: detect runaway repetition and recover from last valid balanced position.
    # VLM repetition looks like the same short phrase appearing 10+ times — truncate at
    # the start of the repetition, then walk backwards to find a balanced JSON prefix.
    import re
    # Find a ~20-char substring that appears 5+ times in a row
    rep_match = re.search(r"(.{5,50}?)\1{4,}", cleaned)
    if rep_match:
        trunc_at = rep_match.start()
        trimmed = cleaned[:trunc_at]
        # Walk backwards and try to close open braces/quotes to make valid JSON
        # Simple approach: keep shrinking from the end until json.loads succeeds,
        # appending closing chars as needed.
        for i in range(len(trimmed), max(0, len(trimmed) - 5000), -1):
            candidate = trimmed[:i].rstrip(",\n \t")
            # Try padding with closing braces/quotes in different combos
            for suffix in ('"}', '"}]}', '"]}', '}', ']}', '"}}', '""}]}'):
                try:
                    return json.loads(candidate + suffix), None
                except json.JSONDecodeError:
                    continue
        return None, f"runaway repetition detected but recovery failed"

    return None, f"invalid JSON: {first_err_msg}"


print(f"Helper functions ready (VLM_MODE={VLM_MODE}).")


### Extraction prompt and assembly function

This cell defines two things:

1. **`LIBRARY_PROMPT`** — the per-page prompt sent to the VLM. Captures
   bibliographic metadata, full verbatim transcription, structural elements
   (headings, footnotes, verse), visual elements (illustrations, musical
   notation, maps), marginalia, stamps/bookplates, physical condition, and
   continuation flags for cross-page content.

2. **`assemble_library_doc()`** — aggregates per-page VLM output into the
   document-level JSON. Picks best bibliographic values across pages,
   concatenates body_text with page markers, deduplicates stamps/marks,
   collects marginalia and illustrations with page references, and runs
   programmatic cross-page merging from continuation flags.


### Per-page extraction prompt (library/archival) (editable in its own cell)

Edit `LIBRARY_PROMPT` below to tweak what the VLM is asked to extract per page. Re-run this cell after changes — the batch loop uses `LIBRARY_PROMPT` directly.


In [ ]:
# ── Library / archival prompt (per-page) ─────────────────────────────
# For scanned books, manuscripts, sheet music, newspapers, maps,
# photographs, pamphlets, and other archival materials.
# Primary goal: FAITHFUL verbatim transcription of the page text.
# Secondary: structured metadata on top (page_type, bibliographic, etc.).

LIBRARY_PROMPT = """Role & Objective: You are an expert OCR TRANSCRIPTION assistant for digitized library and archival materials. Your PRIMARY job is a faithful, verbatim transcription of every word on this scanned page. Metadata fields on top (page_type, bibliographic, summaries) are secondary — transcription fidelity comes first.

Task: Extract all information from this single scanned page into the JSON structure below. body_text is the most important field and must contain the complete page text verbatim.

{
  "body_text": "<COMPLETE VERBATIM transcription of EVERY WORD OF PRINTED OR TYPED TEXT on the page, in reading order. This includes text that is stylized, decorative, set in fancy typography, integrated with illustrations, or appears on a title page. If you can read a word, it goes here. Preserve original line breaks, paragraph breaks, spacing, spelling, punctuation, capitalization, and diacritics EXACTLY as they appear. This is the core OCR output — it must NOT be a summary, paraphrase, or abridgement. If body_text ends up empty, you have failed the task.>",

  "page_type": "<book_page | title_page | table_of_contents | index | manuscript | sheet_music | newspaper | map | photograph | illustration | plate | form | correspondence | ephemera | blank | mixed>",

  "page_number": "<printed, stamped, or handwritten page/folio number>",
  "header": "<running header text if present>",

  "bibliographic": {
    "title": "<running title, chapter title, article headline, or piece title visible on this page>",
    "creator": "<author, composer, artist, editor, or cartographer if shown>",
    "date": "<any date visible: publication, manuscript, postmark, etc.>",
    "language": "<primary language — ISO 639-1 code if clear, else spell out>",
    "other_languages": ["<additional languages if multilingual>"],
    "publisher": "<publisher or printer if shown>",
    "place": "<place of publication or origin>"
  },

  "structure": {
    "headings": ["<chapter titles, section headings, article headlines as they appear>"],
    "footnotes": ["<footnote or endnote text, preserving reference marks>"],
    "lists": ["<any enumerated or bulleted lists>"],
    "poetry_verse": "<verse text preserving original line breaks, or null>"
  },

  "tables": [
    {
      "caption": "<table caption if any>",
      "table_data": "<2D array of cell values, or key-value object for forms>"
    }
  ],

  "visual_elements": {
    "illustrations": [
      {"description": "<factual description of what is depicted — NO interpretation>", "caption": "<printed caption verbatim>", "position": "<top|bottom|left|right|center|full_page>"}
    ],
    "musical_notation": {
      "present": false,
      "instrument_voice": "<e.g. Piano, Soprano, Violin I>",
      "key_signature": "<e.g. G major, C minor>",
      "time_signature": "<e.g. 4/4, 3/4, 6/8>",
      "tempo_markings": "<e.g. Allegro, Andante>",
      "dynamics": ["<pp, mf, crescendo, etc.>"],
      "lyrics": "<sung text under notation, preserving syllable breaks>",
      "description": "<general description of what is notated>"
    },
    "maps": {"present": false, "region": "", "description": ""}
  },

  "marginalia": [
    {"text": "<transcription or [illegible]>", "location": "<top|bottom|left|right|interlinear>", "writing_instrument": "<pencil|pen|ink|stamp>"}
  ],

  "stamps_marks": [
    {"text": "<verbatim stamp text>", "type": "<library_stamp | bookplate | accession_number | call_number | barcode | ownership_mark | censor_mark | other>"}
  ],

  "physical_condition": {
    "damage": "<describe any tears, stains, foxing, fading, water damage, missing portions>",
    "legibility": "<fully_legible | mostly_legible | partially_legible | poor>",
    "scan_issues": "<skew, shadow from binding, cropped edges, bleed-through from verso>"
  },

  "one_sentence_summary": "<brief description of what this page contains — ONE sentence only>",
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "<brief note on transcription quality>",

  "continuation": {
    "continues_from_previous": false,
    "continues_to_next": false,
    "continuation_type": "<null | text | table | verse | musical_notation | list>"
  },

  "other_metadata": {}
}

TRANSCRIPTION RULES (HIGHEST PRIORITY):
- body_text MUST contain the full page text word-for-word. Do NOT summarize, paraphrase, abridge, or skip text. If the page has 500 words of body text, body_text contains all 500 words.
- Preserve original spelling exactly — DO NOT modernize. Old spellings, archaic forms, dialectal variants, unusual capitalization: keep as-is.
- Preserve original punctuation, line breaks, paragraph structure.
- Preserve diacritics, accents, ligatures (æ, œ, ß, etc.).
- For Fraktur / Gothic / blackletter script: transcribe using modern Latin equivalents (ſ → s, etc.) but keep all other orthography.
- Illegible word → write [illegible]. Uncertain reading → write the best guess followed by [?]. Missing/cut-off text → [...].
- Do NOT fill in what "should" be there based on context. Transcribe what you SEE.
- Marginalia (handwritten notes in margins) are NOT part of body_text — extract them separately into marginalia.
- Stamps and library marks are NOT part of body_text — extract them separately into stamps_marks.
- Foreign/multilingual text: transcribe as-is in the original language. Set language/other_languages accordingly.
- TEXT WITHIN ILLUSTRATIONS: if an illustration on the page has text as part of its design (title page calligraphy, captions integrated with artwork, text framing images), that text STILL GOES IN body_text. The illustrations field describes the VISUAL content; any readable text belongs in body_text regardless of how it is styled or positioned.
- TITLE PAGES, COVERS, COLOPHONS: every legible word goes in body_text even if the page is mostly decorative. Do NOT leave body_text empty because the page "looks like a design" — transcribe the words.

METADATA RULES:
- page_type: best single classification for this page.
- bibliographic: only what's visible ON THIS PAGE. Leave empty if not visible.
- Illustrations: describe factually ("woodcut of a shepherd with sheep") — do NOT interpret symbolism.
- Musical notation: describe the printed notation (instrument, key, tempo). Transcribe lyrics if present. Do not encode notes.
- one_sentence_summary: literally one sentence about the page content.
- confidence_narrative: brief (1 sentence) note on scan/transcription quality.

DO NOT:
- Replace body_text with a summary of the page.
- Modernize spelling or punctuation.
- Interpret or analyze content — just transcribe.
- Skip text because it looks repetitive or decorative.
- Invent content to fill empty fields — use null/empty if not present.
- Leave body_text empty because the page is stylized, decorative, or mostly an illustration. If there are readable words, transcribe them.

Output ONLY valid JSON. No markdown fences, no introductory text."""


print(f"Library prompt ready ({len(LIBRARY_PROMPT)} chars).")


In [ ]:
# ── Library / archival prompt (per-page) ─────────────────────────────
# For scanned books, manuscripts, sheet music, newspapers, maps,
# photographs, pamphlets, and other archival materials.
# Handles multilingual text, musical notation, marginalia, stamps.

# ── Pass 1 / pass 2 split for library pipeline ────────────────────────
# Pass 1 = aggregated per-page data (no cross-page mutations)
# Pass 2 = + programmatic cross-page merging + LLM synthesis

def assemble_library_pass1_raw(filename, page_results, model_name, prompt_text=None, run_info=None):
    """Pass 1 library output — aggregated per-page data, no cross-page mutations.

    Includes all the PascalCase aggregations (Bibliographic, BodyText,
    Headings, Illustrations, Marginalia, StampsMarks, etc.) but does NOT
    stitch content across page boundaries. Cross-page merging lives in
    build_library_cross_page_merges() called at pass 2 time.
    """
    from datetime import datetime

    def _best_value(field_name, subdict_key="bibliographic"):
        for pr in page_results:
            val = pr.get("extracted", {}).get(subdict_key, {}).get(field_name)
            if val and str(val).strip():
                return val
        return ""

    bibliographic = {
        "Title": _best_value("title"),
        "Creator": _best_value("creator"),
        "Date": _best_value("date"),
        "Language": _best_value("language"),
        "OtherLanguages": [],
        "Publisher": _best_value("publisher"),
        "Place": _best_value("place"),
    }
    seen_langs = set()
    for pr in page_results:
        for l in pr.get("extracted", {}).get("bibliographic", {}).get("other_languages", []) or []:
            if l and l not in seen_langs:
                seen_langs.add(l)
                bibliographic["OtherLanguages"].append(l)

    body_parts = []
    page_types = []
    summaries = []
    all_headings, all_footnotes, all_lists, all_verse = [], [], [], []
    all_illustrations, all_marginalia, all_stamps = [], [], []
    musical_pages, map_pages = [], []
    physical_issues = []
    all_tables = []
    confidence_scores = []
    page_confidences = []

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})

        page_types.append({"PageNumber": pg, "PageType": d.get("page_type", "unknown")})
        summaries.append({"PageNumber": pg, "Summary": d.get("one_sentence_summary") or "N/A"})

        bt = d.get("body_text", "") or ""
        if bt:
            body_parts.append(f"[PAGE {pg}]\n{bt}")

        for h in d.get("structure", {}).get("headings", []) or []:
            if h:
                all_headings.append({"PageNumber": pg, "Heading": h})
        for fn in d.get("structure", {}).get("footnotes", []) or []:
            if fn:
                all_footnotes.append({"PageNumber": pg, "Footnote": fn})
        for lst in d.get("structure", {}).get("lists", []) or []:
            if lst:
                all_lists.append({"PageNumber": pg, "List": lst})
        verse = d.get("structure", {}).get("poetry_verse")
        if verse:
            all_verse.append({"PageNumber": pg, "Verse": verse})

        for t in d.get("tables", []) or []:
            all_tables.append({
                "PageNumber": pg,
                "Caption": t.get("caption", ""),
                "TableData": t.get("table_data", []),
            })

        for ill in d.get("visual_elements", {}).get("illustrations", []) or []:
            all_illustrations.append({
                "PageNumber": pg,
                "Description": ill.get("description", ""),
                "Caption": ill.get("caption", ""),
                "Position": ill.get("position", ""),
            })
        mn = d.get("visual_elements", {}).get("musical_notation", {})
        if mn and mn.get("present"):
            musical_pages.append({
                "PageNumber": pg,
                "InstrumentVoice": mn.get("instrument_voice", ""),
                "KeySignature": mn.get("key_signature", ""),
                "TimeSignature": mn.get("time_signature", ""),
                "TempoMarkings": mn.get("tempo_markings", ""),
                "Dynamics": mn.get("dynamics", []),
                "Lyrics": mn.get("lyrics", ""),
                "Description": mn.get("description", ""),
            })
        mp = d.get("visual_elements", {}).get("maps", {})
        if mp and mp.get("present"):
            map_pages.append({
                "PageNumber": pg,
                "Region": mp.get("region", ""),
                "Description": mp.get("description", ""),
            })

        for m in d.get("marginalia", []) or []:
            if m and m.get("text"):
                all_marginalia.append({
                    "PageNumber": pg,
                    "Text": m.get("text", ""),
                    "Location": m.get("location", ""),
                    "WritingInstrument": m.get("writing_instrument", ""),
                })

        for st in d.get("stamps_marks", []) or []:
            if st and st.get("text"):
                all_stamps.append({
                    "PageNumber": pg,
                    "Text": st.get("text", ""),
                    "Type": st.get("type", "other"),
                })

        pc = d.get("physical_condition", {})
        if pc and (pc.get("damage") or pc.get("scan_issues") or pc.get("legibility") not in ("fully_legible", None, "")):
            physical_issues.append({
                "PageNumber": pg,
                "Damage": pc.get("damage", ""),
                "Legibility": pc.get("legibility", ""),
                "ScanIssues": pc.get("scan_issues", ""),
            })

        conf_val = d.get("confidence_percentage")
        if isinstance(conf_val, (int, float)):
            confidence_scores.append(conf_val)
        page_confidences.append({
            "PageNumber": pg,
            "ConfidencePercentage": conf_val if isinstance(conf_val, (int, float)) else 0,
            "ConfidenceNarrative": d.get("confidence_narrative", "") if isinstance(d.get("confidence_narrative"), str) else "",
        })

    # Dedup stamps by (text, type) — this is per-page aggregation, not cross-page mutation
    stamp_dedup = {}
    for s in all_stamps:
        key = (s["Text"], s["Type"])
        if key not in stamp_dedup:
            stamp_dedup[key] = {"Text": s["Text"], "Type": s["Type"], "Pages": []}
        stamp_dedup[key]["Pages"].append(s["PageNumber"])
    deduped_stamps = list(stamp_dedup.values())

    avg_conf = round(sum(confidence_scores) / len(confidence_scores), 1) if confidence_scores else 0.0
    low_conf = [pc for pc in page_confidences if pc["ConfidencePercentage"] < 80]
    if low_conf:
        problem_pages = sorted(set(pc["PageNumber"] for pc in low_conf))
        confidence_summary = (
            f"{len(page_results)} pages processed, avg confidence {avg_conf}%. "
            f"{len(low_conf)} page(s) below 80% confidence. Problem pages: {problem_pages}."
        )
    else:
        confidence_summary = (
            f"{len(page_results)} pages processed, avg confidence {avg_conf}%. "
            f"All pages extracted successfully."
        )

    page_times_ms = [pr.get("elapsed_ms", 0) for pr in page_results if pr.get("elapsed_ms")]
    timing = {
        "TotalElapsedSec": round(sum(page_times_ms) / 1000, 1) if page_times_ms else 0,
        "AvgPageSec": round(sum(page_times_ms) / len(page_times_ms) / 1000, 2) if page_times_ms else 0,
        "MinPageSec": round(min(page_times_ms) / 1000, 2) if page_times_ms else 0,
        "MaxPageSec": round(max(page_times_ms) / 1000, 2) if page_times_ms else 0,
    }

    return {
        "Filename": filename,
        "PageCount": len(page_results),
        "RunInfo": {
            "Model": model_name,
            "Timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
            "Timing": timing,
            **(run_info or {}),
        },
        "ExtractionPrompt": prompt_text,
        "ConfidencePercentage": avg_conf,
        "ConfidenceSummary": confidence_summary,
        "PageConfidences": page_confidences,
        "PageResults": page_results,
        "Bibliographic": bibliographic,
        "PageTypes": page_types,
        "OneSentenceNarrativeSummary": summaries,
        "BodyText": "\n\n".join(body_parts),
        "Headings": all_headings,
        "Footnotes": all_footnotes,
        "Lists": all_lists,
        "Verse": all_verse,
        "TablesCollection": all_tables,
        "Illustrations": all_illustrations,
        "MusicalNotation": musical_pages,
        "Maps": map_pages,
        "Marginalia": all_marginalia,
        "StampsMarks": deduped_stamps,
        "PhysicalCondition": physical_issues,
    }


def build_library_cross_page_merges(page_results):
    """Pass 2 ONLY: cross-page merging for library docs.

    Currently just produces CrossPageLinks (metadata only). Future work
    could add body_text stitching across continuation chains, but that's
    fragile enough that we leave it out for now.
    """
    def _cont_flag(d, key):
        v = d.get("continuation", {}).get(key)
        if isinstance(v, bool):
            return v
        if isinstance(v, str):
            return v.strip().lower() == "true"
        return False

    cross_page_links = []
    for i, pr in enumerate(page_results):
        d = pr.get("extracted", {})
        cont = d.get("continuation", {})
        if cont.get("continues_to_next"):
            next_pg = page_results[i + 1]["page"] if i + 1 < len(page_results) else None
            cross_page_links.append({
                "from_page": pr["page"],
                "to_page": next_pg,
                "type": cont.get("continuation_type", "unknown"),
            })

    return {
        "CrossPageLinks": cross_page_links,
    }


# Backwards-compat alias
def assemble_library_doc(filename, page_results, model_name, prompt_text=None, run_info=None):
    """DEPRECATED backward-compat alias."""
    pass1 = assemble_library_pass1_raw(filename, page_results, model_name, prompt_text, run_info)
    merges = build_library_cross_page_merges(page_results)
    return {**pass1, **merges}


print("Library prompt and assembly function ready.")


## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/" + f.name)

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Render pages

All pages are sent as images to the VLM (captures layout, tables, signatures, watermarks).

In [ ]:
import fitz
from PIL import Image

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_images = []
page_links = []
for i, page in enumerate(doc):
    mat = fitz.Matrix(1.0, 1.0)
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    page_images.append(img)

    links = extract_links(page)
    page_links.append(links)

    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    label = "digital" if has_text else "scanned"
    link_info = f", {len(links)} link(s)" if links else ""
    print(f"Page {i+1}: {img.width}x{img.height} ({label}, {len(text)} chars{link_info})")

doc.close()
print(f"\n{len(page_images)} page(s) rendered. All will be sent as images to VLM.")


### Test extraction on a single page

Quick sanity check — renders one page and sends it to the VLM. Change `PAGE_IDX` to test different pages. Review the raw JSON output in the next cell before running the full batch.


In [ ]:
PAGE_IDX = 0  # Change this to test different pages
img = page_images[PAGE_IDX]

# Sliding window context (adjacent pages)
prev_img = page_images[PAGE_IDX - 1] if PAGE_IDX > 0 else None
next_img = page_images[PAGE_IDX + 1] if PAGE_IDX < len(page_images) - 1 else None
ctx_label = f" (with context: pages {PAGE_IDX if prev_img else ''}{'–' if prev_img and next_img else ''}{PAGE_IDX+2 if next_img else ''})" if (prev_img or next_img) else ""

print(f"Page {PAGE_IDX+1}: {img.width}x{img.height}{ctx_label}")
display(img.resize((img.width // 3, img.height // 3)))

t0 = time.time()
raw_result = extract_page(img, LIBRARY_PROMPT, filename=DOC_PATH.name,
                          links=page_links[PAGE_IDX],
                          prev_image=prev_img, next_image=next_img)
elapsed = time.time() - t0
print(f"Extraction took {elapsed:.1f}s")


### Inspect the VLM's JSON output

Parses the raw VLM response (strips markdown fences if present) and pretty-prints the structured JSON. Check that tables, narratives, stakeholders etc. are being extracted correctly before running the batch.


In [ ]:
import json

# Use the robust parser (handles markdown fences, embedded prose, etc.)
parsed, err = parse_vlm_json(raw_result)

if err:
    print(f"Invalid JSON: {err}\n")
    print("Raw output:")
    print(raw_result)
else:
    # ── Human-readable preview for this single page ──
    print("=" * 70)
    print(f"PAGE PREVIEW")
    print("=" * 70)

    pt = parsed.get("page_type", "unknown")
    conf = parsed.get("confidence_percentage", "?")
    print(f"Page type:    {pt}")
    print(f"Confidence:   {conf}%")

    summary = parsed.get("one_sentence_summary", "")
    if summary:
        print(f"Summary:      {summary}")

    bib = parsed.get("bibliographic", {}) or {}
    title = bib.get("title") or ""
    creator = bib.get("creator") or ""
    date = bib.get("date") or ""
    lang = bib.get("language") or ""
    other_langs = bib.get("other_languages") or []
    if title or creator or date or lang:
        print("-" * 70)
        if title:   print(f"Title:        {title}")
        if creator: print(f"Creator:      {creator}")
        if date:    print(f"Date:         {date}")
        if lang:    print(f"Language:     {lang}" + (f"  (also: {', '.join(other_langs)})" if other_langs else ""))

    pn = parsed.get("page_number") or ""
    header = parsed.get("header") or ""
    if pn:     print(f"Page number:  {pn}")
    if header: print(f"Header:       {header}")

    body = parsed.get("body_text") or ""
    if body:
        print("-" * 70)
        print("BODY TEXT:")
        print(body)

    struct = parsed.get("structure", {}) or {}
    headings = struct.get("headings") or []
    footnotes = struct.get("footnotes") or []
    lists_ = struct.get("lists") or []
    verse = struct.get("poetry_verse")
    if headings:
        print("-" * 70)
        print("HEADINGS:")
        for h in headings:
            print(f"  • {h}")
    if footnotes:
        print("-" * 70)
        print("FOOTNOTES:")
        for fn in footnotes:
            print(f"  • {fn}")
    if lists_:
        print("-" * 70)
        print("LISTS:")
        for lst in lists_:
            print(f"  • {lst}")
    if verse:
        print("-" * 70)
        print("VERSE:")
        print(verse)

    vis = parsed.get("visual_elements", {}) or {}
    ills = vis.get("illustrations") or []
    if ills:
        print("-" * 70)
        print("ILLUSTRATIONS:")
        for ill in ills:
            desc = ill.get("description", "")
            cap = ill.get("caption", "")
            pos = ill.get("position", "")
            print(f"  • [{pos}] {desc}" + (f' — caption: "{cap}"' if cap else ""))

    mn = vis.get("musical_notation", {}) or {}
    if mn.get("present"):
        print("-" * 70)
        print("MUSICAL NOTATION:")
        for k in ("instrument_voice", "key_signature", "time_signature", "tempo_markings"):
            if mn.get(k): print(f"  {k}: {mn[k]}")
        if mn.get("dynamics"): print(f"  dynamics: {', '.join(mn['dynamics'])}")
        if mn.get("lyrics"):   print(f"  lyrics: {mn['lyrics']}")
        if mn.get("description"): print(f"  description: {mn['description']}")

    mp = vis.get("maps", {}) or {}
    if mp.get("present"):
        print("-" * 70)
        print(f"MAP: {mp.get('region', '')} — {mp.get('description', '')}")

    marg = parsed.get("marginalia") or []
    if marg:
        print("-" * 70)
        print("MARGINALIA:")
        for m in marg:
            loc = m.get("location", "")
            inst = m.get("writing_instrument", "")
            print(f'  • [{loc}, {inst}] "{m.get("text", "")}"')

    stamps = parsed.get("stamps_marks") or []
    if stamps:
        print("-" * 70)
        print("STAMPS / MARKS:")
        for s in stamps:
            print(f'  • [{s.get("type", "other")}] "{s.get("text", "")}"')

    pc = parsed.get("physical_condition", {}) or {}
    if pc.get("damage") or pc.get("scan_issues") or pc.get("legibility") not in (None, "", "fully_legible"):
        print("-" * 70)
        print("PHYSICAL CONDITION:")
        if pc.get("legibility"):   print(f"  legibility:  {pc['legibility']}")
        if pc.get("damage"):       print(f"  damage:      {pc['damage']}")
        if pc.get("scan_issues"):  print(f"  scan_issues: {pc['scan_issues']}")

    cont = parsed.get("continuation", {}) or {}
    if cont.get("continues_from_previous") or cont.get("continues_to_next"):
        print("-" * 70)
        print(f"CONTINUATION: from_prev={cont.get('continues_from_previous')}, "
              f"to_next={cont.get('continues_to_next')}, type={cont.get('continuation_type')}")

    # Also dump raw JSON at the bottom for reference
    print("\n" + "=" * 70)
    print("RAW JSON")
    print("=" * 70)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))


## 8. Batch process all PDFs in /ocr/

Processes every PDF in the `/ocr/` folder and saves JSONL outputs.

### Configure paths

Set the directories for input PDFs, VLM output, and Gemini reference JSONs. The batch processor will find all `.pdf` files in `PDF_DIR` and save `.jsonl` outputs to `VLM_OUT_DIR`. The comparison step matches files by PDF filename stem.


In [ ]:
import json, fitz
from PIL import Image




def reconstruct_page_results_from_pass1(output):
    """Reconstruct per-page results from an aggregated library pass 1 output."""
    page_count = output.get("PageCount", 0)
    if page_count == 0:
        return []

    by_page = {n: {
        "headings": [],
        "footnotes": [],
        "lists": [],
        "illustrations": [],
        "marginalia": [],
        "stamps_marks": [],
        "tables": [],
        "structure": {"headings": [], "footnotes": [], "lists": [], "poetry_verse": None},
        "visual_elements": {"illustrations": [], "musical_notation": {}, "maps": {}},
        "continuation": {},
    } for n in range(1, page_count + 1)}

    for pc in output.get("PageConfidences", []):
        n = pc.get("PageNumber")
        if n in by_page:
            by_page[n]["confidence_percentage"] = pc.get("ConfidencePercentage", 0)
            by_page[n]["confidence_narrative"] = pc.get("ConfidenceNarrative", "")

    for s in output.get("OneSentenceNarrativeSummary", []):
        if isinstance(s, dict):
            n = s.get("PageNumber")
            if n in by_page:
                by_page[n]["one_sentence_summary"] = s.get("Summary", "")

    for pt in output.get("PageTypes", []):
        n = pt.get("PageNumber")
        if n in by_page:
            by_page[n]["page_type"] = pt.get("PageType", "unknown")

    # Split BodyText back by [PAGE N] markers
    body = output.get("BodyText", "")
    import re
    for match in re.finditer(r"\[PAGE (\d+)\]\n(.*?)(?=\n\n\[PAGE \d+\]|\Z)", body, re.DOTALL):
        n = int(match.group(1))
        if n in by_page:
            by_page[n]["body_text"] = match.group(2).strip()

    # Headings / footnotes / lists / verse with page refs
    for h in output.get("Headings", []):
        n = h.get("PageNumber")
        if n in by_page:
            by_page[n]["structure"]["headings"].append(h.get("Heading", ""))
    for fn in output.get("Footnotes", []):
        n = fn.get("PageNumber")
        if n in by_page:
            by_page[n]["structure"]["footnotes"].append(fn.get("Footnote", ""))
    for lst in output.get("Lists", []):
        n = lst.get("PageNumber")
        if n in by_page:
            by_page[n]["structure"]["lists"].append(lst.get("List", ""))
    for v in output.get("Verse", []):
        n = v.get("PageNumber")
        if n in by_page:
            by_page[n]["structure"]["poetry_verse"] = v.get("Verse", "")

    for ill in output.get("Illustrations", []):
        n = ill.get("PageNumber")
        if n in by_page:
            by_page[n]["visual_elements"]["illustrations"].append({
                "description": ill.get("Description", ""),
                "caption": ill.get("Caption", ""),
                "position": ill.get("Position", ""),
            })

    for mn in output.get("MusicalNotation", []):
        n = mn.get("PageNumber")
        if n in by_page:
            by_page[n]["visual_elements"]["musical_notation"] = {
                "present": True,
                "instrument_voice": mn.get("InstrumentVoice", ""),
                "key_signature": mn.get("KeySignature", ""),
                "time_signature": mn.get("TimeSignature", ""),
                "tempo_markings": mn.get("TempoMarkings", ""),
                "dynamics": mn.get("Dynamics", []),
                "lyrics": mn.get("Lyrics", ""),
                "description": mn.get("Description", ""),
            }

    for mp in output.get("Maps", []):
        n = mp.get("PageNumber")
        if n in by_page:
            by_page[n]["visual_elements"]["maps"] = {
                "present": True,
                "region": mp.get("Region", ""),
                "description": mp.get("Description", ""),
            }

    for m in output.get("Marginalia", []):
        n = m.get("PageNumber")
        if n in by_page:
            by_page[n]["marginalia"].append({
                "text": m.get("Text", ""),
                "location": m.get("Location", ""),
                "writing_instrument": m.get("WritingInstrument", ""),
            })

    for st in output.get("StampsMarks", []):
        for n in st.get("Pages", []):
            if n in by_page:
                by_page[n]["stamps_marks"].append({
                    "text": st.get("Text", ""),
                    "type": st.get("Type", "other"),
                })

    for t in output.get("TablesCollection", []):
        n = t.get("PageNumber")
        if n in by_page:
            by_page[n]["tables"].append({
                "caption": t.get("Caption", ""),
                "table_data": t.get("TableData", []),
            })

    # Bibliographic — attach to page 1 if missing per-page
    if output.get("Bibliographic") and by_page.get(1):
        b = output["Bibliographic"]
        by_page[1]["bibliographic"] = {
            "title": b.get("Title", ""),
            "creator": b.get("Creator", ""),
            "date": b.get("Date", ""),
            "language": b.get("Language", ""),
            "other_languages": b.get("OtherLanguages", []),
            "publisher": b.get("Publisher", ""),
            "place": b.get("Place", ""),
        }

    return [
        {"page": n, "method": "vlm_image", "elapsed_ms": 0, "extracted": by_page[n]}
        for n in range(1, page_count + 1)
    ]


def load_json_or_jsonl(path):
    """Load a JSON or JSONL file. Returns a dict (first object if JSONL/array)."""
    text = path.read_text().strip()
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data[0] if data else {}
        return data
    except json.JSONDecodeError:
        pass
    first_line = text.split("\n")[0].strip()
    data = json.loads(first_line)
    if isinstance(data, list):
        return data[0] if data else {}
    return data

# ── Configurable paths ────────────────────────────────────────────────
PDF_DIR = Path("/ocr")                          # PDFs to process
VLM_OUT_DIR = Path("/ocr/vlm_output")           # VLM JSONL output
GEMINI_DIR = Path("/ocr/gemini")                # Gemini reference JSONs

VLM_OUT_DIR.mkdir(exist_ok=True)

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s) in {PDF_DIR}")
for p in pdf_files:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

if GEMINI_DIR.exists():
    gemini_files = sorted(GEMINI_DIR.glob("*.json")) + sorted(GEMINI_DIR.glob("*.jsonl"))
    print(f"\nFound {len(gemini_files)} Gemini JSON(s) in {GEMINI_DIR}")
    for g in gemini_files:
        print(f"  {g.name}")
else:
    print(f"\nGemini dir not found: {GEMINI_DIR}")
    print("Create it and upload Gemini JSONs to enable comparison.")


### Pass 2: Document-level synthesis helpers

Defines `run_pass2_synthesis()` and `save_final_output()`. These are called
**inline per-doc during the batch loop below**, so you see progress as each
document completes.

**Output files per doc:**
- `{filename}_extracted.jsonl` — raw pass 1, kept as audit trail
- `{filename}_final.json` — consolidated canonical output (pass 1 content +
  code-driven cross-page merges + pass 2 synthesis block)

Pass 2 produces document-level metadata: title, type, creator, date, summary,
cross-page notes. The LLM synthesis is purely additive — it does not modify
any pass 1 content. Cross-page table/narrative merging is still done
programmatically (no LLM) in the assembly function.

**Set `RUN_PASS2 = False`** in the batch cell to skip pass 2 (no _final.json
is produced; only _extracted.jsonl).
**Set `RERUN_PASS2 = True`** in this cell to re-run synthesis over an
existing `batch_results` list (e.g. after tweaking `SYNTHESIS_PROMPT`).

**Token budget:** `max_context_tokens=14000` by default (for vLLM servers
with `--max-model-len 16384`). Bump to `28000` if you bump the server to
`--max-model-len 32768`. The function prints a warning when input is ≥75%
of budget, and skips pass 2 for docs that exceed the budget.


### Pass 2 catalog synthesis prompt (library/archival) (editable in its own cell)

Edit `SYNTHESIS_PROMPT` below to tweak what the VLM is asked to extract per page. Re-run this cell after changes — the batch loop uses `SYNTHESIS_PROMPT` directly.


In [ ]:
SYNTHESIS_PROMPT = """You are given the per-page extraction results from a digitized library document.
Each page was processed independently by a VLM. Your job is to produce
ONLY a document-level cataloging summary. Do NOT rewrite, merge, or modify
the per-page content — it is the source of truth.

Return a JSON object with catalog-style metadata:
{
  "document_title": "<best title found across all pages (usually title page)>",
  "document_type": "<book | manuscript | periodical_issue | newspaper | sheet_music | atlas | map | correspondence | pamphlet | ephemera | mixed>",
  "creator": "<author, composer, editor, cartographer, etc.>",
  "creator_role": "<author | composer | editor | translator | cartographer | illustrator | photographer | unknown>",
  "date": "<most relevant date: publication, manuscript, postmark, etc.>",
  "publisher": "<publisher or printer>",
  "place_of_publication": "<city, country>",
  "primary_language": "<ISO 639-1 or language name>",
  "other_languages": ["<additional languages present>"],
  "subject_tags": ["<2-8 subject tags describing content, e.g. 'American Civil War', 'Hymnody', 'Botany'>"],
  "document_summary": "<3-4 sentence summary synthesizing across ALL pages — what is this document, what does it contain, notable features>",
  "collection_notes": {
    "call_number": "<if visible on any page>",
    "accession_number": "<if visible>",
    "isbn_issn": "<if printed work>",
    "provenance": "<summary of stamps/bookplates/ownership marks seen>"
  },
  "physical_description_summary": "<overall condition summary: binding, damage patterns, legibility, notable scan artifacts>",
  "notable_features": ["<things a cataloger would flag: illustrations, musical notation, marginalia, multilingual sections, missing pages, etc.>"],
  "cross_page_notes": ["<observations about content spanning pages, e.g. 'Chapter 2 begins p.12', 'Multi-page musical score pp.45-48'>"]
}

RULES:
- Extract bibliographic metadata from whichever page contains it (usually title page).
- For document_summary, synthesize across ALL pages — reference specific content from multiple pages.
- For subject_tags, use Library of Congress-style tags if possible.
- For collection_notes, aggregate all stamps/bookplates seen into a single provenance narrative.
- For cross_page_notes, look at continuation flags in each page's JSON.
- Output ONLY valid JSON. No markdown fences."""

In [ ]:
from pathlib import Path

# ── Pass 2: Document-level synthesis ─────────────────────────────────
# Reads per-page JSONs (text only, no images) and produces document-level
# metadata. Uses the same VLM/LLM — just a text-in, text-out call.
#
# This is OPTIONAL — pass 1 output is complete without it.

def _compact_page_for_synthesis(d):
    """Light compact: drop redundant raw_*_text + context_snippet.

    These fields just duplicate info already in the structured fields
    (full_name, email, address_line1, etc.). No information loss.
    """
    kept = {k: v for k, v in d.items() if k not in ("stakeholders", "addresses")}
    if d.get("stakeholders"):
        kept["stakeholders"] = [
            {k: v for k, v in s.items()
             if k not in ("raw_stakeholder_text", "context_snippet")
             and v not in (None, "", [])}
            for s in d["stakeholders"] if any(s.values())
        ]
    if d.get("addresses"):
        kept["addresses"] = [
            {k: v for k, v in a.items()
             if k not in ("raw_address_text", "context_snippet")
             and v not in (None, "", [])}
            for a in d["addresses"] if any(a.values())
        ]
    return kept


# Fields that are page-local noise for doc-level synthesis (boilerplate,
# already rolled up programmatically, or binary flags of little use).
SYNTHESIS_NOISE_FIELDS = (
    "confidence_narrative",
    "continuation",
    "has_annotation",
    "has_watermark",
    "other_metadata",
    "signature_lines",
)


def _drop_noise_fields(d):
    """Drop per-page fields that don't help document-level synthesis."""
    kept = _compact_page_for_synthesis(d)
    for f in SYNTHESIS_NOISE_FIELDS:
        kept.pop(f, None)
    return kept


def _dedup_structural(page_results_list):
    """Return a new list of page_results with stakeholders/addresses
    deduplicated across pages.

    If the same stakeholder (matched by full_name + institution) or address
    (matched by address_line1 + city + postal_code) appears on multiple
    pages, only the first occurrence is kept. Later pages still keep other
    fields — just not duplicates of stakeholders/addresses already seen.
    """
    seen_stakeholders = set()
    seen_addresses = set()
    out = []
    for pr in page_results_list:
        d = dict(pr.get("extracted", {}))  # shallow copy

        if d.get("stakeholders"):
            unique = []
            for s in d["stakeholders"]:
                key = (
                    (s.get("full_name") or "").strip().lower(),
                    (s.get("institution") or "").strip().lower(),
                    (s.get("stakeholder_role") or "").strip().lower(),
                )
                if key not in seen_stakeholders:
                    seen_stakeholders.add(key)
                    unique.append(s)
            d["stakeholders"] = unique

        if d.get("addresses"):
            unique = []
            for a in d["addresses"]:
                key = (
                    (a.get("address_line1") or "").strip().lower(),
                    (a.get("city") or "").strip().lower(),
                    (a.get("postal_code") or "").strip().lower(),
                )
                if key not in seen_addresses:
                    seen_addresses.add(key)
                    unique.append(a)
            d["addresses"] = unique

        new_pr = dict(pr)
        new_pr["extracted"] = d
        out.append(new_pr)
    return out


def _trim_narratives_page(d, snippet_chars=None, max_narratives=None):
    """Trim narrative_responses: cap per-entry verbatim_text + cap count per page."""
    kept = dict(d)
    if kept.get("narrative_responses"):
        narrs = kept["narrative_responses"]
        if max_narratives is not None:
            narrs = narrs[:max_narratives]
        if snippet_chars is not None:
            trimmed = []
            for n in narrs:
                text = n.get("verbatim_text", "") or ""
                snippet = text[:snippet_chars]
                entry = {
                    "prompt_or_header": n.get("prompt_or_header", ""),
                    "verbatim_text": snippet,
                }
                if len(text) > snippet_chars:
                    entry["verbatim_text"] += "…"
                    entry["full_length_chars"] = len(text)
                trimmed.append(entry)
            narrs = trimmed
        kept["narrative_responses"] = narrs
    return kept


def _trim_tables_page(d, sample_rows=None):
    """Trim tables: cap row count, keep classification + column headers + sample."""
    kept = dict(d)
    if kept.get("tables"):
        trimmed = []
        for t in kept["tables"]:
            data = t.get("table_data")
            classification = t.get("table_classification", "Standard_Table")
            entry = {"table_classification": classification}
            if isinstance(data, list) and data:
                entry["row_count"] = len(data)
                if sample_rows is None:
                    entry["table_data"] = data
                elif sample_rows > 0:
                    entry["sample_rows"] = data[:sample_rows]
                if classification == "Standard_Table" and isinstance(data[0], dict):
                    entry["column_headers"] = list(data[0].keys())
            elif isinstance(data, dict):
                entry["table_data"] = data
            else:
                entry["table_data"] = data
            trimmed.append(entry)
        kept["tables"] = trimmed
    return kept


def run_pass2_synthesis(page_results, model_name, max_context_tokens=31000):
    # Note: with current --max-model-len 16384, budget is 14K.
    # If you bump the vLLM server to 32K, raise max_context_tokens to ~28K.
    """Run pass 2 synthesis over per-page JSON results (text-only, no images).

    Uses a compact per-page representation so a 20+ page document fits in
    a 16K max_model_len server. If estimated tokens still exceed the budget,
    warns and skips.

    Returns the synthesis dict, or None if it fails.
    """
    def _build(page_results_list, compact_fn=None):
        parts = []
        for pr in page_results_list:
            parts.append(f"--- PAGE {pr['page']} ---")
            d = pr.get("extracted", {}) if compact_fn is None else compact_fn(pr.get("extracted", {}))
            parts.append(json.dumps(d, indent=1, ensure_ascii=False))
        return "\n".join(parts)

    # Start: raw per-page JSON (no compaction at all)
    all_pages = _build(page_results, None)
    est_tokens = len(all_pages) // 4
    tier_used = "raw"

    if est_tokens > max_context_tokens:
        # Cascade through progressively more aggressive compaction.
        # Trim order: tables BEFORE narratives, since tables contribute less
        # to a 3-sentence document summary than narrative prose does.
        #
        # 1. Light compact (drop raw_*_text, context_snippet)
        # 2. + Drop synthesis-noise fields (confidence_narrative, continuation,
        #    has_annotation, has_watermark, signature_lines, other_metadata)
        # 3. + Dedup stakeholders/addresses across pages
        # 4. + Tables: sample 2 rows
        # 5. + Tables: sample 1 row
        # 6. + Tables: classification + column headers only (no rows)
        # 7. + Narratives @ 400 chars
        # 8. + Narratives @ 200 chars
        # 9. + Narratives @ 100 chars, cap 8/page
        # 10. + Narratives @ 60 chars, cap 4/page

        deduped_results = None  # lazy — compute when needed

        def _try(pages_list, compact_fn, label):
            nonlocal all_pages, est_tokens, tier_used
            built = _build(pages_list, compact_fn)
            est = len(built) // 4
            if est <= max_context_tokens:
                print(f"  [pass2] {label} → {est} tokens (under {max_context_tokens} budget)")
                all_pages = built
                est_tokens = est
                tier_used = label
                return True
            return False

        # Step 1: light compact
        if _try(page_results, _compact_page_for_synthesis,
                f"light compact (raw was {est_tokens})"):
            pass
        # Step 2: + drop noise fields
        elif _try(page_results, _drop_noise_fields, "+ drop noise fields"):
            pass
        else:
            # Steps 3+: dedup structural fields across pages
            if deduped_results is None:
                deduped_results = _dedup_structural(page_results)

            # Step 3: dedup + noise drop
            if _try(deduped_results, _drop_noise_fields,
                    "+ dedup stakeholders/addresses"):
                pass
            else:
                # Steps 4-6: trim TABLES first (tables contribute less to synthesis)
                table_knobs = [
                    {"sample_rows": 2, "label": "+ tables@2rows"},
                    {"sample_rows": 1, "label": "+ tables@1row"},
                    {"sample_rows": 0, "label": "+ tables: headers only"},
                ]
                fit = False
                for k in table_knobs:
                    def _fn(d, _k=k):
                        return _trim_tables_page(_drop_noise_fields(d),
                                                  sample_rows=_k["sample_rows"])
                    if _try(deduped_results, _fn, k["label"]):
                        fit = True
                        break

                if not fit:
                    # Steps 7-10: tables at headers-only + progressively trim NARRATIVES
                    narrative_knobs = [
                        {"snippet_chars": 400, "max_narratives": None, "label": "+ narratives@400ch"},
                        {"snippet_chars": 200, "max_narratives": None, "label": "+ narratives@200ch"},
                        {"snippet_chars": 100, "max_narratives": 8,    "label": "+ narratives@100ch×8"},
                        {"snippet_chars": 60,  "max_narratives": 4,    "label": "+ narratives@60ch×4"},
                    ]
                    for k in narrative_knobs:
                        def _fn(d, _k=k):
                            stage1 = _trim_tables_page(_drop_noise_fields(d), sample_rows=0)
                            return _trim_narratives_page(stage1,
                                                          snippet_chars=_k["snippet_chars"],
                                                          max_narratives=_k["max_narratives"])
                        if _try(deduped_results, _fn, k["label"]):
                            break

    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{all_pages}"}
        ]}
    ]

    try:
        raw = run_vlm(messages, max_tokens=1500)
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(cleaned)
    except Exception as e:
        print(f"  [pass2] Synthesis failed: {e}")
        return None



def render_library_markdown(final_doc):
    """Render a library/archival final doc as human-readable markdown."""
    lines = []
    lines.append(f"# {final_doc.get('Filename', 'Unknown')}")
    lines.append("")

    synth = final_doc.get("DocumentSynthesis", {})
    bib = final_doc.get("Bibliographic", {})

    # Header card — prefer synthesis fields, fall back to bibliographic
    title = synth.get("document_title") or bib.get("Title", "—")
    creator = synth.get("creator") or bib.get("Creator", "—")
    date = synth.get("date") or bib.get("Date", "—")
    doc_type = synth.get("document_type", "—")
    publisher = synth.get("publisher") or bib.get("Publisher", "")
    place = synth.get("place_of_publication") or bib.get("Place", "")
    lang = synth.get("primary_language") or bib.get("Language", "")

    lines.append(f"**Type:** {doc_type}")
    lines.append(f"**Title:** {title}")
    lines.append(f"**Creator:** {creator}")
    if synth.get("creator_role"):
        lines.append(f"**Creator role:** {synth['creator_role']}")
    lines.append(f"**Date:** {date}")
    if publisher:
        lines.append(f"**Publisher:** {publisher}")
    if place:
        lines.append(f"**Place:** {place}")
    lines.append(f"**Language:** {lang}")
    other_langs = bib.get("OtherLanguages", []) or synth.get("other_languages", [])
    if other_langs:
        lines.append(f"**Other languages:** {', '.join(other_langs)}")
    lines.append("")

    lines.append(f"**Pages:** {final_doc.get('PageCount', 0)}")
    lines.append(f"**Confidence:** {final_doc.get('ConfidencePercentage', 0)}%")
    ri = final_doc.get("RunInfo", {})
    lines.append(f"**Model:** {ri.get('Model', '—')}")
    tm = ri.get("Timing", {})
    if tm:
        lines.append(f"**Timing:** {tm.get('TotalElapsedSec', 0)}s total, {tm.get('AvgPageSec', 0)}s avg/page")
    lines.append("")

    # Subject tags
    if synth.get("subject_tags"):
        lines.append(f"**Subject tags:** {', '.join(synth['subject_tags'])}")
        lines.append("")

    # Summary
    if synth.get("document_summary"):
        lines.append("## Summary")
        lines.append(synth["document_summary"])
        lines.append("")

    # Collection notes (call number, provenance, etc.)
    cn = synth.get("collection_notes", {})
    if cn and any(cn.values()):
        lines.append("## Collection / catalog notes")
        for k, v in cn.items():
            if v:
                lines.append(f"- **{k}:** {v}")
        lines.append("")

    # Physical description
    if synth.get("physical_description_summary"):
        lines.append("## Physical description")
        lines.append(synth["physical_description_summary"])
        lines.append("")

    # Notable features
    if synth.get("notable_features"):
        lines.append("## Notable features")
        for f in synth["notable_features"]:
            lines.append(f"- {f}")
        lines.append("")

    # Cross-page notes
    if synth.get("cross_page_notes"):
        lines.append("## Cross-page notes")
        for note in synth["cross_page_notes"]:
            lines.append(f"- {note}")
        lines.append("")

    # Low-confidence pages
    low_pages = [pc for pc in final_doc.get("PageConfidences", []) if pc.get("ConfidencePercentage", 100) < 80]
    if low_pages:
        lines.append("## ⚠️ Low-confidence pages")
        for pc in low_pages:
            lines.append(f"- **Page {pc['PageNumber']}** ({pc['ConfidencePercentage']}%): {pc.get('ConfidenceNarrative', '')}")
        lines.append("")

    # Headings (mini TOC)
    headings = final_doc.get("Headings", [])
    if headings:
        lines.append("## Table of contents (from headings)")
        for h in headings:
            lines.append(f"- p.{h['PageNumber']} — {h.get('Heading', '')}")
        lines.append("")

    # Stamps & marks (provenance)
    stamps = final_doc.get("StampsMarks", [])
    if stamps:
        lines.append("## Library stamps / bookplates / marks")
        for s in stamps:
            pages = ", ".join(str(p) for p in s.get("Pages", []))
            lines.append(f"- **{s.get('Type', 'other')}** (pages {pages}): \"{s.get('Text', '')}\"")
        lines.append("")

    # Illustrations
    ills = final_doc.get("Illustrations", [])
    if ills:
        lines.append("## Illustrations")
        for ill in ills:
            cap = ill.get("Caption", "")
            desc = ill.get("Description", "")
            pos = ill.get("Position", "")
            lines.append(f"- p.{ill['PageNumber']} ({pos}): {desc}" + (f' — caption: \"{cap}\"' if cap else ""))
        lines.append("")

    # Musical notation
    mns = final_doc.get("MusicalNotation", [])
    if mns:
        lines.append("## Musical notation")
        for mn in mns:
            lines.append(f"### Page {mn['PageNumber']}")
            bits = []
            for k in ("InstrumentVoice", "KeySignature", "TimeSignature", "TempoMarkings"):
                if mn.get(k):
                    bits.append(f"**{k}:** {mn[k]}")
            if bits:
                lines.append(" | ".join(bits))
            if mn.get("Dynamics"):
                lines.append(f"**Dynamics:** {', '.join(mn['Dynamics'])}")
            if mn.get("Lyrics"):
                lines.append(f"**Lyrics:** {mn['Lyrics']}")
            if mn.get("Description"):
                lines.append(mn["Description"])
            lines.append("")

    # Marginalia
    marg = final_doc.get("Marginalia", [])
    if marg:
        lines.append("## Marginalia")
        for m in marg:
            loc = m.get("Location", "")
            inst = m.get("WritingInstrument", "")
            lines.append(f"- p.{m['PageNumber']} ({loc}, {inst}): \"{m.get('Text', '')}\"")
        lines.append("")

    # Physical condition issues
    pc_issues = final_doc.get("PhysicalCondition", [])
    if pc_issues:
        lines.append("## Physical condition issues")
        for issue in pc_issues:
            bits = []
            for k in ("Damage", "ScanIssues"):
                if issue.get(k):
                    bits.append(f"{k}: {issue[k]}")
            if bits:
                lines.append(f"- p.{issue['PageNumber']}: {'; '.join(bits)}")
        lines.append("")

    # Body text — the actual transcription
    body_text = final_doc.get("BodyText", "")
    lines.append("## Full transcription (BodyText)")
    lines.append("")
    if body_text and body_text.strip():
        # body_text already has [PAGE N] markers — render as-is in a code block so formatting holds
        lines.append("```")
        lines.append(body_text)
        lines.append("```")
    else:
        lines.append("> ⚠️ **No body text extracted for this document.**")
        lines.append(">")
        lines.append("> This can happen when the VLM treated all page text as part of an illustration/caption, or when the pages are truly blank.")
        lines.append("> Check the per-page extractions in the _extracted.json file (PageResults) to see what was captured.")

    return "\n".join(lines)


def save_final_output(out_path, pass1_output, synthesis, pass2_elapsed):
    """Save the consolidated final library doc JSON."""
    from datetime import datetime
    final_path = str(out_path).replace("_extracted.jsonl", "_final.json").replace("_extracted.json", "_final.json")

    page_results = pass1_output.get("PageResults", [])
    merges = build_library_cross_page_merges(page_results)

    final = {**pass1_output, **merges}

    final["RunInfo"] = {
        **pass1_output.get("RunInfo", {}),
        "Pass2ElapsedSec": round(pass2_elapsed, 1),
        "Pass2Timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        "SourceExtractionFile": Path(out_path).name,
    }

    final["DocumentSynthesis"] = synthesis
    final["SynthesisPrompt"] = SYNTHESIS_PROMPT

    with open(final_path, "w") as f:
        json.dump(final, f, indent=2, ensure_ascii=False)

    # Also write human-readable markdown rendering for QA/debug
    md_path = final_path.replace("_final.json", "_readable.md")
    try:
        Path(md_path).write_text(render_library_markdown(final), encoding="utf-8")
    except Exception as e:
        print(f"  (readable.md render failed: {e})")

    return final_path


print("Pass 2 functions ready: run_pass2_synthesis(), save_final_output()")
print("Pass 2 now runs inline in the batch loop — no need to run this cell separately")
print("(Cell kept for optional re-runs over an existing batch_results list)")

# Optional: re-run pass 2 over batch_results if needed (e.g. after tweaking prompt)
RERUN_PASS2 = False
if RERUN_PASS2 and batch_results:
    print(f"\nRe-running pass 2 on {len(batch_results)} documents...\n")
    for br in batch_results:
        if "page_results" not in br:
            continue
        t0 = time.time()
        synthesis = run_pass2_synthesis(br["page_results"], MODEL_NAME)
        elapsed = time.time() - t0
        if synthesis:
            br["synthesis"] = synthesis
            if br.get("output_path") and br.get("output"):
                p = save_final_output(br["output_path"], br["output"], synthesis, elapsed)
                print(f"  {br['filename']}: OK ({elapsed:.1f}s) -> {Path(p).name}")
        else:
            print(f"  {br['filename']}: FAILED ({elapsed:.1f}s)")


### Run batch extraction

Loops through every PDF in the input directory. For each document:
1. Renders all pages at 1x resolution
2. Sends each page image to the VLM with the extraction prompt
3. Parses the JSON response (handles markdown fences and parse failures)
4. Assembles page results into a document-level JSONL and saves to disk

Progress is printed per-page with timing.


In [ ]:
import json, fitz, time
from pathlib import Path
from PIL import Image

SKIP_EXISTING_PASS1 = True  # Skip pass 1 if _extracted.jsonl already exists
SKIP_EXISTING_PASS2 = True  # Skip pass 2 if _final.json already exists
USE_SLIDING_WINDOW = True   # 3-page context per call
RUN_PASS2 = True            # Run document-level synthesis

batch_results = []
pass1_skipped = 0
pass2_skipped = 0
pass2_resumed = 0

for pdf_path in pdf_files:
    # New: use .json extension (we write a single pretty-printed JSON object, not JSONL).
    # Jupyter was failing to render .jsonl because it expected one JSON per line.
    out_path = VLM_OUT_DIR / f"{pdf_path.stem}_extracted.json"
    # Back-compat: if an older .jsonl file exists, read from it
    legacy_jsonl = VLM_OUT_DIR / f"{pdf_path.stem}_extracted.jsonl"
    if legacy_jsonl.exists() and not out_path.exists():
        out_path = legacy_jsonl
    final_path = VLM_OUT_DIR / f"{pdf_path.stem}_final.json"

    pass1_exists = out_path.exists()
    pass2_exists = final_path.exists()

    # Case 1: fully done — skip entirely
    if SKIP_EXISTING_PASS1 and pass1_exists and (not RUN_PASS2 or (SKIP_EXISTING_PASS2 and pass2_exists)):
        pass1_skipped += 1
        if pass2_exists:
            pass2_skipped += 1
        output = load_json_or_jsonl(out_path)
        batch_results.append({"filename": pdf_path.name, "output": output,
                              "output_path": str(out_path)})
        print(f"SKIP (done): {pdf_path.name}")
        continue

    # Case 2: pass 1 done, pass 2 missing — resume at pass 2
    if SKIP_EXISTING_PASS1 and pass1_exists and RUN_PASS2 and not pass2_exists:
        print(f"\n{'='*60}")
        print(f"Resuming pass 2 only: {pdf_path.name}")
        print(f"{'='*60}")
        pass1_skipped += 1
        pass2_resumed += 1

        output = load_json_or_jsonl(out_path)
        # Recover page_results from the saved pass 1 output.
        # Newer files have PageResults embedded; older files fall back to
        # reconstructing from the aggregated fields by grouping on PageNumber.
        page_results = output.get("PageResults")
        if not page_results:
            print(f"  (older format — reconstructing page_results from aggregated fields)")
            page_results = reconstruct_page_results_from_pass1(output)
        if not page_results:
            print(f"  WARNING: could not reconstruct page_results — re-running pass 1")
        else:
            t2 = time.time()
            synthesis = run_pass2_synthesis(page_results, MODEL_NAME)
            elapsed2 = time.time() - t2
            br = {"filename": pdf_path.name, "output": output,
                  "page_results": page_results, "output_path": str(out_path)}
            if synthesis:
                br["synthesis"] = synthesis
                fp = save_final_output(str(out_path), output, synthesis, elapsed2)
                print(f"  Final saved: {Path(fp).name} ({elapsed2:.1f}s) — {synthesis.get('document_type', '?')}")
            else:
                print(f"  Pass 2 FAILED ({elapsed2:.1f}s)")
            batch_results.append(br)
            continue
        # fall through to full reprocess if PageResults missing

    # Case 3: fresh processing
    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")

    doc = fitz.open(str(pdf_path))
    page_count = len(doc)

    images = []
    links_per_page = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(1.0, 1.0)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
        links_per_page.append(extract_links(page))
    doc.close()
    mode = "sliding window (3-page)" if USE_SLIDING_WINDOW else "single page"
    print(f"  {page_count} page(s) rendered — extraction mode: {mode}")

    page_results = []
    for i, img in enumerate(images):
        page_num = i + 1
        t0 = time.time()

        if USE_SLIDING_WINDOW:
            prev_img = images[i - 1] if i > 0 else None
            next_img = images[i + 1] if i < len(images) - 1 else None
            raw = extract_page(img, LIBRARY_PROMPT, filename=pdf_path.name,
                               links=links_per_page[i],
                               prev_image=prev_img, next_image=next_img)
        else:
            raw = extract_page(img, LIBRARY_PROMPT, filename=pdf_path.name,
                               links=links_per_page[i])

        elapsed = time.time() - t0

        extracted, parse_err = parse_vlm_json(raw)
        if extracted is None:
            print(f"    Page {page_num}: PARSE FAILED ({parse_err})")
            debug_dir = VLM_OUT_DIR / "parse_failures"
            debug_dir.mkdir(exist_ok=True)
            debug_path = debug_dir / f"{pdf_path.stem}_page{page_num:03d}_raw.txt"
            debug_path.write_text(raw, encoding="utf-8")
            print(f"      Raw output saved: {debug_path.name}")
            extracted = {
                "raw_text": raw,
                "parse_error": parse_err,
                "confidence_percentage": 0,
                "confidence_narrative": f"Failed to parse structured output ({parse_err})",
            }

        page_results.append({"page": page_num, "method": "vlm_image",
                             "elapsed_ms": round(elapsed * 1000, 1), "extracted": extracted})
        ctx = "+ctx" if USE_SLIDING_WINDOW else ""
        print(f"    Page {page_num}/{page_count}{ctx}: {elapsed:.1f}s")

    # Assemble and save pass 1 output (now includes PageResults for resumability)
    run_info = {
        "VLMMode": VLM_MODE,
        "VLLMEndpoint": VLLM_BASE_URL if VLM_MODE == "remote" else None,
        "UseSlidingWindow": USE_SLIDING_WINDOW,
    }
    output = assemble_library_pass1_raw(pdf_path.name, page_results, MODEL_NAME,
                                         prompt_text=LIBRARY_PROMPT, run_info=run_info)
    out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False) + "\n")
    print(f"  Pass 1 saved: {out_path.name}")

    br = {"filename": pdf_path.name, "output": output,
          "page_results": page_results, "output_path": str(out_path)}

    if RUN_PASS2:
        t2 = time.time()
        synthesis = run_pass2_synthesis(page_results, MODEL_NAME)
        elapsed2 = time.time() - t2
        if synthesis:
            br["synthesis"] = synthesis
            final_path_saved = save_final_output(str(out_path), output, synthesis, elapsed2)
            print(f"  Final saved: {Path(final_path_saved).name} ({elapsed2:.1f}s) — {synthesis.get('document_type', '?')}")
        else:
            print(f"  Pass 2 FAILED ({elapsed2:.1f}s) — no final file (audit _extracted.jsonl still saved)")

    batch_results.append(br)

print(f"\n{'='*60}")
print(f"Batch complete. {len(batch_results)} doc(s) total.")
print(f"  Pass 1 skipped (already done): {pass1_skipped}")
print(f"  Pass 2 skipped (already done): {pass2_skipped}")
print(f"  Pass 2 resumed (pass 1 existed, pass 2 missing): {pass2_resumed}")
print(f"  Fresh processed: {len(batch_results) - pass1_skipped}")
print(f"Output dir: {VLM_OUT_DIR}")


## 9. Compare VLM vs Gemini extractions

Side-by-side comparison of what our local VLM extracted vs what Gemini produced for the same PDFs. This tells us whether the local model is competitive with Gemini for grant document extraction.

**How it works:**
1. Matches files by PDF name (e.g. `foo.pdf` -> VLM's `foo_extracted.jsonl` vs Gemini's `foo.json`)
2. For each matched pair, compares every section of the output schema
3. Produces a per-document text report + 6-panel visual comparison

**What the metrics mean:**

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Narrative text coverage** | Total characters of verbatim text extracted | More = the model captured more of the actual document content. Critical for RAG — missing text means missing search results. |
| **Text similarity** | How much the VLM and Gemini narrative text overlap (0-100%) | High = both models extracted the same content. Low = one model found text the other missed, OR they structured it differently. |
| **Table cells** | Number of individual data cells extracted from tables | More cells = more complete table extraction. Missing cells = lost dollar amounts, dates, etc. |
| **Stakeholders** | People/orgs identified (PIs, contacts, sponsors) | Checks if the model finds all named individuals and their roles. |
| **Addresses** | Physical addresses found in the document | Checks if letterhead, mailing, and signature block addresses are captured. |
| **Confidence** | The model's self-reported extraction confidence | Self-assessed, so take with a grain of salt. Useful for flagging pages the model struggled with. |
| **Wins scorecard** | Which model "won" more categories across all documents | Quick summary: is VLM or Gemini better overall? |


### Comparison functions

Defines the comparison and plotting code. Run this cell to load the functions — the next cell actually executes the comparison.


In [ ]:
import json
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import numpy as np

def compare_extractions(vlm, gem, filename):
    """Compare VLM vs Gemini extraction for a single document. Returns report dict."""
    report = {"filename": filename, "sections": {}}

    # ── Narrative coverage
    vlm_narr = vlm.get("NarrativeResponses", [])
    gem_narr = gem.get("NarrativeResponses", [])
    vlm_text = " ".join(n.get("VerbatimText", "") for n in vlm_narr)
    gem_text = " ".join(n.get("VerbatimText", "") for n in gem_narr)
    text_sim = SequenceMatcher(None, vlm_text.lower(), gem_text.lower()).ratio() if (vlm_text and gem_text) else 0

    report["sections"]["NarrativeResponses"] = {
        "vlm_count": len(vlm_narr), "gemini_count": len(gem_narr),
        "vlm_total_chars": len(vlm_text), "gemini_total_chars": len(gem_text),
        "text_similarity": round(text_sim * 100, 1),
        "vlm_has_citations": "[cite:" in vlm_text,
        "gemini_has_citations": "[cite:" in gem_text,
    }

    # ── Tables
    vlm_tables = vlm.get("TablesCollection", [])
    gem_tables = gem.get("TablesCollection", [])

    def count_table_cells(tables):
        total = 0
        for t in tables:
            td = t.get("TableData", [])
            if isinstance(td, list):
                for row in td:
                    total += len(row) if isinstance(row, (dict, list)) else 1
            elif isinstance(td, dict):
                total += len(td)
        return total

    report["sections"]["TablesCollection"] = {
        "vlm_count": len(vlm_tables), "gemini_count": len(gem_tables),
        "vlm_total_cells": count_table_cells(vlm_tables),
        "gemini_total_cells": count_table_cells(gem_tables),
        "vlm_classifications": [t.get("TableClassification", "?") for t in vlm_tables],
        "gemini_classifications": [t.get("TableClassification", "?") for t in gem_tables],
    }

    # ── Stakeholders
    vlm_stak = vlm.get("Stakeholders", [])
    gem_stak = gem.get("Stakeholders", [])
    vlm_names = sorted(set(s.get("FullName", "") for s in vlm_stak if s.get("FullName")))
    gem_names = sorted(set(s.get("FullName", "") for s in gem_stak if s.get("FullName")))

    report["sections"]["Stakeholders"] = {
        "vlm_count": len(vlm_stak), "gemini_count": len(gem_stak),
        "vlm_names": vlm_names, "gemini_names": gem_names,
        "names_in_common": sorted(set(vlm_names) & set(gem_names)),
        "vlm_only": sorted(set(vlm_names) - set(gem_names)),
        "gemini_only": sorted(set(gem_names) - set(vlm_names)),
    }

    # ── Addresses
    vlm_addr = vlm.get("AddressesCollection", [])
    gem_addr = gem.get("AddressesCollection", [])
    report["sections"]["AddressesCollection"] = {
        "vlm_count": len(vlm_addr), "gemini_count": len(gem_addr),
    }

    # ── Document tags
    vlm_tags = set(vlm.get("DocumentTags", []))
    gem_tags = set(gem.get("DocumentTags", []))
    report["sections"]["DocumentTags"] = {
        "vlm_tags": sorted(vlm_tags), "gemini_tags": sorted(gem_tags),
        "in_common": sorted(vlm_tags & gem_tags),
        "vlm_only": sorted(vlm_tags - gem_tags),
        "gemini_only": sorted(gem_tags - vlm_tags),
    }

    # ── Document details
    vlm_dd = vlm.get("DocumentDetails", {})
    gem_dd = gem.get("DocumentDetails", {})
    dd_diffs = {}
    for key in set(list(vlm_dd.keys()) + list(gem_dd.keys())):
        v, g = vlm_dd.get(key, ""), gem_dd.get(key, "")
        if str(v).strip() != str(g).strip():
            dd_diffs[key] = {"vlm": v, "gemini": g}

    report["sections"]["DocumentDetails"] = {
        "matching_fields": len(vlm_dd) - len(dd_diffs),
        "differing_fields": dd_diffs,
    }

    # ── Flags
    report["sections"]["Flags"] = {
        "HasAnnotation": {"vlm": vlm.get("HasAnnotation"), "gemini": gem.get("HasAnnotation")},
        "HasWatermark": {"vlm": vlm.get("HasWatermark"), "gemini": gem.get("HasWatermark")},
        "SignatureLines": {"vlm": vlm.get("SignatureLines"), "gemini": gem.get("SignatureLines")},
    }

    # ── Confidence
    report["sections"]["Confidence"] = {
        "vlm": vlm.get("ConfidencePercentage"),
        "gemini": gem.get("ConfidencePercentage"),
        "vlm_narrative": vlm.get("ConfidenceNarrative", "")[:200],
        "gemini_narrative": gem.get("ConfidenceNarrative", "")[:200],
    }

    # ── FileNameMetaData
    vlm_fm = vlm.get("FileNameMetaData", {})
    gem_fm = gem.get("FileNameMetaData", {})
    fm_match = sum(1 for k in vlm_fm if str(vlm_fm.get(k, "")) == str(gem_fm.get(k, "")))

    report["sections"]["FileNameMetaData"] = {
        "matching": fm_match, "total": max(len(vlm_fm), len(gem_fm)),
        "diffs": {k: {"vlm": vlm_fm.get(k, ""), "gemini": gem_fm.get(k, "")}
                  for k in set(list(vlm_fm.keys()) + list(gem_fm.keys()))
                  if str(vlm_fm.get(k, "")) != str(gem_fm.get(k, ""))},
    }

    return report


def print_report(report):
    """Pretty-print a comparison report."""
    print(f"\n{'='*70}")
    print(f"  {report['filename']}")
    print(f"{'='*70}")
    s = report["sections"]

    n = s["NarrativeResponses"]
    print(f"\n  NARRATIVES")
    print(f"    Sections:   VLM={n['vlm_count']}, Gemini={n['gemini_count']}")
    print(f"    Text chars: VLM={n['vlm_total_chars']:,}, Gemini={n['gemini_total_chars']:,}")
    print(f"    Similarity: {n['text_similarity']}%")
    print(f"    Citations:  VLM={'yes' if n['vlm_has_citations'] else 'NO'}, "
          f"Gemini={'yes' if n['gemini_has_citations'] else 'NO'}")
    winner = "VLM" if n["vlm_total_chars"] >= n["gemini_total_chars"] else "Gemini"
    print(f"    >> More text coverage: {winner}")

    t = s["TablesCollection"]
    print(f"\n  TABLES")
    print(f"    Count: VLM={t['vlm_count']}, Gemini={t['gemini_count']}")
    print(f"    Cells: VLM={t['vlm_total_cells']}, Gemini={t['gemini_total_cells']}")

    sk = s["Stakeholders"]
    print(f"\n  STAKEHOLDERS")
    print(f"    Count: VLM={sk['vlm_count']}, Gemini={sk['gemini_count']}")
    if sk["names_in_common"]: print(f"    Common:      {sk['names_in_common']}")
    if sk["vlm_only"]: print(f"    VLM only:    {sk['vlm_only']}")
    if sk["gemini_only"]: print(f"    Gemini only: {sk['gemini_only']}")

    a = s["AddressesCollection"]
    print(f"\n  ADDRESSES: VLM={a['vlm_count']}, Gemini={a['gemini_count']}")

    dd = s["DocumentDetails"]
    print(f"\n  DOCUMENT DETAILS: {dd['matching_fields']} matching")
    for k, v in dd["differing_fields"].items():
        print(f"    {k}: VLM={v['vlm']!r} vs Gemini={v['gemini']!r}")

    c = s["Confidence"]
    print(f"\n  CONFIDENCE: VLM={c['vlm']}%, Gemini={c['gemini']}%")

    fm = s["FileNameMetaData"]
    print(f"  FILENAME METADATA: {fm['matching']}/{fm['total']} matching")


def plot_comparison(reports, model_name):
    """Generate comparison plots for VLM vs Gemini."""
    if not reports:
        print("No reports to plot.")
        return

    doc_names = [r["filename"][:40] for r in reports]
    n_docs = len(reports)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"VLM ({model_name.split('/')[-1]}) vs Gemini", fontsize=14, fontweight="bold")

    colors_vlm = "#2196F3"
    colors_gem = "#FF9800"

    # 1. Narrative text coverage (chars)
    ax = axes[0, 0]
    vlm_chars = [r["sections"]["NarrativeResponses"]["vlm_total_chars"] for r in reports]
    gem_chars = [r["sections"]["NarrativeResponses"]["gemini_total_chars"] for r in reports]
    x = np.arange(n_docs)
    w = 0.35
    ax.barh(x - w/2, vlm_chars, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_chars, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Characters")
    ax.set_title("Narrative Text Coverage")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 2. Text similarity
    ax = axes[0, 1]
    sims = [r["sections"]["NarrativeResponses"]["text_similarity"] for r in reports]
    bars = ax.barh(x, sims, color=["#4CAF50" if s >= 70 else "#FFC107" if s >= 40 else "#F44336" for s in sims])
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Similarity %")
    ax.set_title("Narrative Text Similarity")
    ax.set_xlim(0, 100)
    ax.axvline(70, color="gray", linestyle="--", alpha=0.5)
    ax.invert_yaxis()

    # 3. Table cells
    ax = axes[0, 2]
    vlm_cells = [r["sections"]["TablesCollection"]["vlm_total_cells"] for r in reports]
    gem_cells = [r["sections"]["TablesCollection"]["gemini_total_cells"] for r in reports]
    ax.barh(x - w/2, vlm_cells, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_cells, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Cells extracted")
    ax.set_title("Table Extraction (cells)")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 4. Stakeholders + Addresses
    ax = axes[1, 0]
    vlm_sk = [r["sections"]["Stakeholders"]["vlm_count"] for r in reports]
    gem_sk = [r["sections"]["Stakeholders"]["gemini_count"] for r in reports]
    vlm_ad = [r["sections"]["AddressesCollection"]["vlm_count"] for r in reports]
    gem_ad = [r["sections"]["AddressesCollection"]["gemini_count"] for r in reports]
    w2 = 0.2
    ax.barh(x - 1.5*w2, vlm_sk, w2, label="VLM Stakeholders", color=colors_vlm)
    ax.barh(x - 0.5*w2, gem_sk, w2, label="Gemini Stakeholders", color=colors_gem)
    ax.barh(x + 0.5*w2, vlm_ad, w2, label="VLM Addresses", color=colors_vlm, alpha=0.5)
    ax.barh(x + 1.5*w2, gem_ad, w2, label="Gemini Addresses", color=colors_gem, alpha=0.5)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Count")
    ax.set_title("Stakeholders & Addresses")
    ax.legend(fontsize=7)
    ax.invert_yaxis()

    # 5. Confidence scores
    ax = axes[1, 1]
    vlm_conf = [r["sections"]["Confidence"]["vlm"] or 0 for r in reports]
    gem_conf = [r["sections"]["Confidence"]["gemini"] or 0 for r in reports]
    ax.barh(x - w/2, vlm_conf, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_conf, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Confidence %")
    ax.set_title("Self-Reported Confidence")
    ax.set_xlim(0, 100)
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 6. Wins scorecard
    ax = axes[1, 2]
    categories = ["Narratives", "Tables", "Stakeholders", "Addresses"]
    vlm_w = [0, 0, 0, 0]
    gem_w = [0, 0, 0, 0]
    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]: vlm_w[0] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]: gem_w[0] += 1
        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]: vlm_w[1] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]: gem_w[1] += 1
        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]: vlm_w[2] += 1
        elif sk["gemini_count"] > sk["vlm_count"]: gem_w[2] += 1
        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]: vlm_w[3] += 1
        elif a["gemini_count"] > a["vlm_count"]: gem_w[3] += 1
    y = np.arange(len(categories))
    ax.barh(y - w/2, vlm_w, w, label="VLM wins", color=colors_vlm)
    ax.barh(y + w/2, gem_w, w, label="Gemini wins", color=colors_gem)
    ax.set_yticks(y)
    ax.set_yticklabels(categories)
    ax.set_xlabel("Documents won")
    ax.set_title("Wins Scorecard")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

print("Comparison functions ready (with plots).")


### Run comparison and generate plots

Runs the comparison for all matched VLM/Gemini pairs and prints:
- **Per-document report**: detailed breakdown of each section (narratives, tables, stakeholders, etc.)
- **Aggregate scorecard**: tallies which model wins each category across all documents
- **6-panel chart**: visual comparison — narrative coverage, text similarity, table completeness, stakeholders/addresses, confidence, and overall wins


In [ ]:
# ── Run comparison ────────────────────────────────────────────────────
reports = []
matched = 0
unmatched_vlm = []
unmatched_gem = []

# Build lookup: PDF stem -> VLM output
vlm_files = {p.stem.replace("_extracted", ""): p
             for p in VLM_OUT_DIR.glob("*.jsonl")}

# Build lookup: PDF stem -> Gemini output (try multiple naming conventions)
gem_lookup = {}
if GEMINI_DIR.exists():
    for g in list(GEMINI_DIR.glob("*.json")) + list(GEMINI_DIR.glob("*.jsonl")):
        stem = g.stem.replace("_extracted", "").replace("_gemini", "")
        gem_lookup[stem] = g

# Match and compare
for stem, vlm_path in sorted(vlm_files.items()):
    if stem in gem_lookup:
        matched += 1
        vlm_data = load_json_or_jsonl(vlm_path)
        gem_data = load_json_or_jsonl(gem_lookup[stem])
        report = compare_extractions(vlm_data, gem_data, vlm_data.get("Filename", stem))
        reports.append(report)
        print_report(report)
    else:
        unmatched_vlm.append(stem)

for stem in gem_lookup:
    if stem not in vlm_files:
        unmatched_gem.append(stem)

# ── Aggregate summary ─────────────────────────────────────────────────
print(f"\n\n{'#'*70}")
print(f"  AGGREGATE SUMMARY — {len(reports)} document(s) compared")
print(f"{'#'*70}")

if unmatched_vlm:
    print(f"\n  VLM outputs with no Gemini match: {unmatched_vlm}")
if unmatched_gem:
    print(f"  Gemini JSONs with no VLM match: {unmatched_gem}")

if reports:
    # Tally wins per section
    vlm_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    gem_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    total_sim = 0

    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        total_sim += n["text_similarity"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]:
            vlm_wins["narratives"] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]:
            gem_wins["narratives"] += 1

        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]:
            vlm_wins["tables"] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]:
            gem_wins["tables"] += 1

        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]:
            vlm_wins["stakeholders"] += 1
        elif sk["gemini_count"] > sk["vlm_count"]:
            gem_wins["stakeholders"] += 1

        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]:
            vlm_wins["addresses"] += 1
        elif a["gemini_count"] > a["vlm_count"]:
            gem_wins["addresses"] += 1

    n_docs = len(reports)
    print(f"\n  Average narrative text similarity: {total_sim / n_docs:.1f}%")
    print(f"\n  {'Category':<20} {'VLM wins':<12} {'Gemini wins':<12} {'Tie':<8}")
    print(f"  {'-'*52}")
    for cat in vlm_wins:
        ties = n_docs - vlm_wins[cat] - gem_wins[cat]
        print(f"  {cat:<20} {vlm_wins[cat]:<12} {gem_wins[cat]:<12} {ties:<8}")

    total_vlm = sum(vlm_wins.values())
    total_gem = sum(gem_wins.values())
    print(f"  {'-'*52}")
    print(f"  {'TOTAL':<20} {total_vlm:<12} {total_gem:<12} "
          f"{n_docs * 4 - total_vlm - total_gem:<8}")

    if total_vlm > total_gem:
        print(f"\n  >> Overall winner: VLM ({MODEL_NAME})")
    elif total_gem > total_vlm:
        print(f"\n  >> Overall winner: Gemini")
    else:
        print(f"\n  >> Result: TIE")
else:
    print("\n  No matched documents to compare.")
    print("  Upload Gemini JSONs to /ocr/gemini/ with filenames matching the PDFs.")

# ── Plots ─────────────────────────────────────────────────────────────
plot_comparison(reports, MODEL_NAME)


## 10. Launch Streamlit app

Tests the extraction server + Streamlit UI stack from within this notebook — same code that runs in production as separate jobs.

**Prerequisites:**
1. Your workspace must have a **Custom URL tool on port 8501** configured (in addition to Jupyter on 8888). Ask your admin to add this if you don't have it.
2. Set env var `STREAMLIT_BASE_PATH` to the tool's URL path. Check the RunAI UI — click the Custom URL tool link to see the actual path (usually `/<project>/<job-name>/url-1`).
3. The model will be freed from GPU and reloaded by the server process.

**To add the tool:** RunAI UI > your workspace > Edit > Tools > Add Tool > Custom URL, port 8501

**To find the URL path:** RunAI UI > your workspace > Connect > click the Custom URL tool link. The path in the URL after the cluster host is your `STREAMLIT_BASE_PATH`.


In [ ]:
import subprocess

repo_dir = "/ocr/repo"

# Free notebook model from GPU before server loads its own
if 'model' in dir() and model is not None:
    del model
    del processor
    torch.cuda.empty_cache()
    print("Freed notebook model from GPU.")

# Start extraction server in local mode (loads model with transformers, no vLLM)
env = {**os.environ, "LLM_BASE_URL": "local", "HF_HUB_OFFLINE": "1"}
server_proc = subprocess.Popen(
    ["python", "ocr_app/scripts/ocr_server.py"],
    env=env, cwd=repo_dir,
)
print(f"Extraction server starting (PID {server_proc.pid})...")
print("  Mode: LOCAL (transformers, no vLLM)")
print(f"  Model: {MODEL_NAME}")
print("  All pages -> VLM image extraction")

# Wait for server to load model and be ready
import time as _time
for i in range(120):
    _time.sleep(2)
    try:
        import httpx
        resp = httpx.get("http://localhost:8090/health", timeout=2.0)
        if resp.status_code == 200:
            info = resp.json()
            print(f"\nServer ready! LLM: {info.get('llm_model', '?')}")
            break
    except Exception:
        if i % 15 == 14:
            print(f"  Still loading model... ({(i+1)*2}s)")
else:
    print("WARNING: Server did not become ready within 4 min")

# Start Streamlit UI with proxy-compatible base path
base_path = os.environ.get("STREAMLIT_BASE_PATH", "")
streamlit_cmd = [
    "streamlit", "run", "ocr_app/app.py",
    "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true",
]
if base_path:
    streamlit_cmd.append(f"--server.baseUrlPath={base_path}")

streamlit_proc = subprocess.Popen(
    streamlit_cmd,
    env={**os.environ, "OCR_SERVICE_URL": "http://localhost:8090"},
    cwd=repo_dir,
)
print(f"Streamlit started (PID {streamlit_proc.pid})")
if base_path:
    print(f"\nAccess: https://deepthought.doit.wisc.edu{base_path}/")
else:
    print("\nWARNING: STREAMLIT_BASE_PATH not set — proxy URL will 404")
    print("Add env var: STREAMLIT_BASE_PATH = /<project>/<job-name>/url-1")
print("\nTo stop: server_proc.terminate(); streamlit_proc.terminate()")
